# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arjelmilan/flyrank-ai/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Paper Finding 1 — Random Forest feature importance for Health Score

The research paper reports that a Random Forest predicting Health Score placed
Average Position (43%), Impressions (32%), and Scroll Depth (15%) among its
largest feature importances.

#### My methodology question

Where does the label come from, and are the model inputs independent of that label?

The paper defines Health Score using Impressions, Average Position, CTR, and
Scroll Depth. These same variables are then used as predictors in the Random
Forest. Therefore, high feature importance for these variables is expected
because they partly define the target itself.

This does not make the analysis invalid, because the paper explicitly presents
this ML section as exploratory and describes the importance as model behavior
rather than causal evidence. However, I would avoid interpreting these feature
importance values as evidence that these variables independently cause higher
Health Scores.

For my own Week-5 model, I will apply the same check: I will trace exactly how
the target is constructed and verify that no target-derived or overlapping
variables are included as model features.

### Paper Finding 2 — Logistic Regression growth classification

The research paper reports 71% holdout accuracy for a Logistic Regression model
that separates growing from declining pages. Content Age, Days Since Update,
and Days Visible are among the strongest reported signals.

#### My methodology question

Does the validation design support interpreting the measured accuracy as
evidence that the model will generalize to unseen pages or clients?

The paper describes the ML pipeline as using an 80/20 split. An important
question for a repeated-entity dataset is whether related observations from
the same client can appear in both the training and holdout sets.

If the same clients occur in both sets, the holdout accuracy may measure
performance on observations related to clients already represented during
training rather than performance on entirely unseen clients.

I would therefore want to compare the original random split with a grouped
validation design, where all observations from a client remain in either the
training or test set. The difference between the two measurements would itself
be useful evidence about how much the validation design affects the measured
result.

This is a validation question rather than a claim that the reported result is
incorrect. The goal is to understand what population the reported accuracy
actually supports.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

### Before: Random split

In Week 5, I evaluated the Random Forest using a random row-level
train/test split. This measures performance when observations are
randomly divided, but observations from the same client can potentially
appear in both training and test sets.


model_precision@50 = 0.78

### After: Client-grouped split

Week 5 also included a client-level split using `client_hash_id`.
This keeps all observations from a client in either the training or
test set, so the model is evaluated on clients it did not see during
training.

model_precision@50 = 0.64


### Conclusion
The Random Forest achieved higher measured Precision@50 under the random split than under the client-grouped split. The observed gap suggests that the random split may benefit from repeated client-level patterns. The client-grouped result is therefore a more conservative measurement of ranking performance for clients not represented in training.

In [ ]:
import pandas as pd
result = {'validation':['random_split','client_split','difference'],
          'precision@50':[0.78,0.64,0.14]}
df = pd.DataFrame(result)
print(df)

     validation  precision@50
0  random_split          0.78
1  client_split          0.64
2    difference          0.14


## 3. Leakage audit

### 3.1 Label-derived feature audit

The target `is_declining` is calculated using `clk_last15` and
`clk_first15`. Therefore, `clk_last15` is a label-derived variable and
must not be available to the model.

I checked the final feature list and excluded both `clk_last15` and
`is_declining` from the model inputs. The remaining features are based on
information available during the first 15 days of March.

Result: no direct label-derived feature is included in the final feature set.

### 3.2 Future-window audit

I checked the feature and label time windows.

The model features are aggregated from March 1–15, while `clk_last15`,
which is used to construct the target, comes from March 16–31.

Therefore, the feature window does not overlap the label window. The
model does not use the future click outcome when constructing the
first-15-day features.

Result: no temporal overlap was identified between the final feature
window and the label window.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


Observed: The model identifies a measured association with subsequent click decline.

Measured: Performance is evaluated using the tested validation design and ranking metric.

Directional: The result indicates a potential signal for prioritization.

Decision-support: The model can support human review of candidate pages.

Not established: The experiment does not establish that a page objectively needs a refresh or that refreshing it will increase clicks.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.